# Comprehensive Distributed Analysis of Biomass Grinding ML Results

## Overview
This notebook performs a detailed distributed analysis of all machine learning model results to:
1. **Identify the best performing models** across different configurations
2. **Determine the most important features** using multiple methods (SHAP, Feature Importance, Correlation)
3. **Compare different approaches** (Original data vs Imputed data vs Reduced features)
4. **Provide statistical insights** on model performance and feature importance
5. **Generate comprehensive visualizations** for easy interpretation

## Analysis Structure
- **Model Performance Analysis**: R², MAE, RMSE across all configurations
- **Feature Importance Aggregation**: Combining SHAP values, feature importances, and correlations
- **Statistical Analysis**: Distributions, rankings, and significance testing
- **Comparative Visualizations**: Side-by-side comparisons of all approaches


## Step 1: Setup and Data Loading

**Important**: This notebook requires the original `ml_analysis_biomass.ipynb` to be executed first to populate all the model results and feature importance variables.


In [1]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import rankdata
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# Color palette
colors = sns.color_palette("husl", 10)

print("=" * 80)
print("COMPREHENSIVE RESULTS ANALYSIS")
print("=" * 80)
print("\nLibraries imported successfully!")
print("\n⚠️  IMPORTANT: Ensure ml_analysis_biomass.ipynb has been executed completely")
print("    before running this analysis notebook.")


COMPREHENSIVE RESULTS ANALYSIS

Libraries imported successfully!

⚠️  IMPORTANT: Ensure ml_analysis_biomass.ipynb has been executed completely
    before running this analysis notebook.


## Step 2: Aggregate All Model Performance Metrics

This section collects all model performance metrics (R², MAE, RMSE) from the original notebook across:
- **Models**: Random Forest, ANN, XGBoost
- **Datasets**: Original, Imputed
- **Feature Sets**: All features, Spearman reduced, Pearson reduced
- **Targets**: Final Particle Size, Specific Grinding Energy


In [2]:
# Collect all model performance metrics
# This function systematically collects all available metrics from the original notebook

def collect_model_metrics():
    """Collect all model performance metrics from the original notebook."""
    
    metrics = []
    all_vars = globals()
    
    # Define all possible metric combinations
    metric_configs = [
        # Original Data - Target 1
        ('RF', 'Original', 'All', 'Final Particle Size', 'rf_r2_t1', 'rf_mae_t1', 'rf_rmse_t1'),
        ('ANN', 'Original', 'All', 'Final Particle Size', 'ann_r2_t1', 'ann_mae_t1', 'ann_rmse_t1'),
        ('XGBoost', 'Original', 'All', 'Final Particle Size', 'xgb_r2_t1', 'xgb_mae_t1', 'xgb_rmse_t1'),
        # Original Data - Target 2
        ('RF', 'Original', 'All', 'Specific Grinding Energy', 'rf_r2_t2', 'rf_mae_t2', 'rf_rmse_t2'),
        ('ANN', 'Original', 'All', 'Specific Grinding Energy', 'ann_r2_t2', 'ann_mae_t2', 'ann_rmse_t2'),
        ('XGBoost', 'Original', 'All', 'Specific Grinding Energy', 'xgb_r2_t2', 'xgb_mae_t2', 'xgb_rmse_t2'),
        # Spearman Reduced - Target 1
        ('RF', 'Original', 'Spearman Reduced', 'Final Particle Size', 'rf_r2_spearman_t1', 'rf_mae_spearman_t1', 'rf_rmse_spearman_t1'),
        ('ANN', 'Original', 'Spearman Reduced', 'Final Particle Size', 'ann_r2_spearman_t1', 'ann_mae_spearman_t1', 'ann_rmse_spearman_t1'),
        ('XGBoost', 'Original', 'Spearman Reduced', 'Final Particle Size', 'xgb_r2_spearman_t1', 'xgb_mae_spearman_t1', 'xgb_rmse_spearman_t1'),
        # Spearman Reduced - Target 2
        ('RF', 'Original', 'Spearman Reduced', 'Specific Grinding Energy', 'rf_r2_spearman_t2', 'rf_mae_spearman_t2', 'rf_rmse_spearman_t2'),
        ('ANN', 'Original', 'Spearman Reduced', 'Specific Grinding Energy', 'ann_r2_spearman_t2', 'ann_mae_spearman_t2', 'ann_rmse_spearman_t2'),
        ('XGBoost', 'Original', 'Spearman Reduced', 'Specific Grinding Energy', 'xgb_r2_spearman_t2', 'xgb_mae_spearman_t2', 'xgb_rmse_spearman_t2'),
        # Pearson Reduced - Target 1
        ('RF', 'Original', 'Pearson Reduced', 'Final Particle Size', 'rf_r2_pearson_t1', 'rf_mae_pearson_t1', 'rf_rmse_pearson_t1'),
        ('ANN', 'Original', 'Pearson Reduced', 'Final Particle Size', 'ann_r2_pearson_t1', 'ann_mae_pearson_t1', 'ann_rmse_pearson_t1'),
        ('XGBoost', 'Original', 'Pearson Reduced', 'Final Particle Size', 'xgb_r2_pearson_t1', 'xgb_mae_pearson_t1', 'xgb_rmse_pearson_t1'),
        # Pearson Reduced - Target 2
        ('RF', 'Original', 'Pearson Reduced', 'Specific Grinding Energy', 'rf_r2_pearson_t2', 'rf_mae_pearson_t2', 'rf_rmse_pearson_t2'),
        ('ANN', 'Original', 'Pearson Reduced', 'Specific Grinding Energy', 'ann_r2_pearson_t2', 'ann_mae_pearson_t2', 'ann_rmse_pearson_t2'),
        ('XGBoost', 'Original', 'Pearson Reduced', 'Specific Grinding Energy', 'xgb_r2_pearson_t2', 'xgb_mae_pearson_t2', 'xgb_rmse_pearson_t2'),
        # Imputed Data - Target 1
        ('RF', 'Imputed', 'All', 'Final Particle Size', 'rf_r2_imputed_t1', 'rf_mae_imputed_t1', 'rf_rmse_imputed_t1'),
        ('ANN', 'Imputed', 'All', 'Final Particle Size', 'ann_r2_imputed_t1', 'ann_mae_imputed_t1', 'ann_rmse_imputed_t1'),
        ('XGBoost', 'Imputed', 'All', 'Final Particle Size', 'xgb_r2_imputed_t1', 'xgb_mae_imputed_t1', 'xgb_rmse_imputed_t1'),
        # Imputed Data - Target 2
        ('RF', 'Imputed', 'All', 'Specific Grinding Energy', 'rf_r2_imputed_t2', 'rf_mae_imputed_t2', 'rf_rmse_imputed_t2'),
        ('ANN', 'Imputed', 'All', 'Specific Grinding Energy', 'ann_r2_imputed_t2', 'ann_mae_imputed_t2', 'ann_rmse_imputed_t2'),
        ('XGBoost', 'Imputed', 'All', 'Specific Grinding Energy', 'xgb_r2_imputed_t2', 'xgb_mae_imputed_t2', 'xgb_rmse_imputed_t2'),
    ]
    
    for model, dataset, features, target, r2_var, mae_var, rmse_var in metric_configs:
        try:
            r2 = globals().get(r2_var, None)
            mae = globals().get(mae_var, None)
            rmse = globals().get(rmse_var, None)
            
            if r2 is not None and mae is not None and rmse is not None:
                metrics.append({
                    'Model': model,
                    'Target': target,
                    'Dataset': dataset,
                    'Features': features,
                    'R2': float(r2),
                    'MAE': float(mae),
                    'RMSE': float(rmse)
                })
        except:
            pass
    
    return pd.DataFrame(metrics)

# Collect metrics
df_metrics = collect_model_metrics()

print("=" * 80)
print("MODEL PERFORMANCE METRICS COLLECTED")
print("=" * 80)
print(f"\nTotal models analyzed: {len(df_metrics)}")
print(f"\nMetrics DataFrame shape: {df_metrics.shape}")
if len(df_metrics) > 0:
    print("\nFirst few rows:")
    print(df_metrics.head(10))
else:
    print("\n⚠️  No metrics found! Please ensure ml_analysis_biomass.ipynb has been executed.")
    print("    You may need to run the original notebook first.")

MODEL PERFORMANCE METRICS COLLECTED

Total models analyzed: 0

Metrics DataFrame shape: (0, 0)

⚠️  No metrics found! Please ensure ml_analysis_biomass.ipynb has been executed.
    You may need to run the original notebook first.


## Step 3: Model Performance Analysis and Ranking

This section:
- Normalizes metrics for fair comparison
- Calculates composite scores
- Ranks models by performance
- Creates comprehensive visualizations


In [4]:
# Clean and normalize metrics for comparison
df_metrics_clean = df_metrics.dropna(subset=['R2', 'MAE', 'RMSE']).copy()

if len(df_metrics_clean) == 0:
    print("⚠️  No valid metrics found. Please run the original notebook first.")
else:
    # Normalize metrics: R2 (higher is better), MAE/RMSE (lower is better)
    def normalize_metrics(df):
        df_norm = df.copy()
        
        # Normalize R2 (0-1 scale, higher is better)
        r2_min, r2_max = df_norm['R2'].min(), df_norm['R2'].max()
        if r2_max > r2_min:
            df_norm['R2_norm'] = (df_norm['R2'] - r2_min) / (r2_max - r2_min)
        else:
            df_norm['R2_norm'] = 0.5
        
        # Normalize MAE (invert so higher is better)
        mae_min, mae_max = df_norm['MAE'].min(), df_norm['MAE'].max()
        if mae_max > mae_min:
            df_norm['MAE_norm'] = 1 - (df_norm['MAE'] - mae_min) / (mae_max - mae_min)
        else:
            df_norm['MAE_norm'] = 0.5
        
        # Normalize RMSE (invert so higher is better)
        rmse_min, rmse_max = df_norm['RMSE'].min(), df_norm['RMSE'].max()
        if rmse_max > rmse_min:
            df_norm['RMSE_norm'] = 1 - (df_norm['RMSE'] - rmse_min) / (rmse_max - rmse_min)
        else:
            df_norm['RMSE_norm'] = 0.5
        
        # Composite score (weighted average: 50% R2, 25% MAE, 25% RMSE)
        df_norm['Composite_Score'] = (
            0.5 * df_norm['R2_norm'] + 
            0.25 * df_norm['MAE_norm'] + 
            0.25 * df_norm['RMSE_norm']
        )
        
        return df_norm
    
    df_metrics_normalized = normalize_metrics(df_metrics_clean)
    
    print("=" * 80)
    print("MODEL PERFORMANCE ANALYSIS")
    print("=" * 80)
    print(f"\nTotal valid models: {len(df_metrics_normalized)}")
    print(f"\nMetrics summary:")
    print(df_metrics_normalized[['Model', 'Target', 'Dataset', 'Features', 'R2', 'MAE', 'RMSE', 'Composite_Score']].describe())


KeyError: ['R2', 'MAE', 'RMSE']

In [5]:
# Rank models by target
if len(df_metrics_normalized) > 0:
    def rank_models_by_target(df, target_name):
        df_target = df[df['Target'] == target_name].copy()
        df_target = df_target.sort_values('Composite_Score', ascending=False)
        df_target['Rank'] = range(1, len(df_target) + 1)
        return df_target
    
    rankings_t1 = rank_models_by_target(df_metrics_normalized, 'Final Particle Size')
    rankings_t2 = rank_models_by_target(df_metrics_normalized, 'Specific Grinding Energy')
    
    print("=" * 80)
    print("TOP 10 MODELS - FINAL PARTICLE SIZE")
    print("=" * 80)
    display_cols = ['Rank', 'Model', 'Dataset', 'Features', 'R2', 'MAE', 'RMSE', 'Composite_Score']
    print(rankings_t1[display_cols].head(10).to_string(index=False))
    
    print("\n" + "=" * 80)
    print("TOP 10 MODELS - SPECIFIC GRINDING ENERGY")
    print("=" * 80)
    print(rankings_t2[display_cols].head(10).to_string(index=False))
else:
    print("⚠️  No metrics available for ranking.")


NameError: name 'df_metrics_normalized' is not defined

In [ ]:
# Visualize model performance comparison
if len(df_metrics_normalized) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    # Target 1: R² Score
    ax1 = axes[0, 0]
    df_t1 = df_metrics_normalized[df_metrics_normalized['Target'] == 'Final Particle Size']
    if len(df_t1) > 0:
        df_t1_sorted = df_t1.sort_values('R2', ascending=False).head(15)
        ax1.barh(range(len(df_t1_sorted)), df_t1_sorted['R2'], color=colors[:len(df_t1_sorted)])
        ax1.set_yticks(range(len(df_t1_sorted)))
        ax1.set_yticklabels([f"{row['Model']} ({row['Dataset']})" for _, row in df_t1_sorted.iterrows()], fontsize=8)
        ax1.set_xlabel('R² Score', fontweight='bold')
        ax1.set_title('Final Particle Size - R² Score Comparison', fontweight='bold', fontsize=12)
        ax1.grid(axis='x', alpha=0.3)
        ax1.invert_yaxis()
    
    # Target 1: MAE
    ax2 = axes[0, 1]
    if len(df_t1) > 0:
        df_t1_sorted_mae = df_t1.sort_values('MAE', ascending=True).head(15)
        ax2.barh(range(len(df_t1_sorted_mae)), df_t1_sorted_mae['MAE'], color=colors[:len(df_t1_sorted_mae)])
        ax2.set_yticks(range(len(df_t1_sorted_mae)))
        ax2.set_yticklabels([f"{row['Model']} ({row['Dataset']})" for _, row in df_t1_sorted_mae.iterrows()], fontsize=8)
        ax2.set_xlabel('MAE', fontweight='bold')
        ax2.set_title('Final Particle Size - MAE Comparison', fontweight='bold', fontsize=12)
        ax2.grid(axis='x', alpha=0.3)
        ax2.invert_yaxis()
    
    # Target 1: Composite Score
    ax3 = axes[0, 2]
    if len(df_t1) > 0:
        df_t1_sorted_comp = df_t1.sort_values('Composite_Score', ascending=False).head(15)
        ax3.barh(range(len(df_t1_sorted_comp)), df_t1_sorted_comp['Composite_Score'], color=colors[:len(df_t1_sorted_comp)])
        ax3.set_yticks(range(len(df_t1_sorted_comp)))
        ax3.set_yticklabels([f"{row['Model']} ({row['Dataset']})" for _, row in df_t1_sorted_comp.iterrows()], fontsize=8)
        ax3.set_xlabel('Composite Score', fontweight='bold')
        ax3.set_title('Final Particle Size - Composite Score', fontweight='bold', fontsize=12)
        ax3.grid(axis='x', alpha=0.3)
        ax3.invert_yaxis()
    
    # Target 2: R² Score
    ax4 = axes[1, 0]
    df_t2 = df_metrics_normalized[df_metrics_normalized['Target'] == 'Specific Grinding Energy']
    if len(df_t2) > 0:
        df_t2_sorted = df_t2.sort_values('R2', ascending=False).head(15)
        ax4.barh(range(len(df_t2_sorted)), df_t2_sorted['R2'], color=colors[:len(df_t2_sorted)])
        ax4.set_yticks(range(len(df_t2_sorted)))
        ax4.set_yticklabels([f"{row['Model']} ({row['Dataset']})" for _, row in df_t2_sorted.iterrows()], fontsize=8)
        ax4.set_xlabel('R² Score', fontweight='bold')
        ax4.set_title('Specific Grinding Energy - R² Score Comparison', fontweight='bold', fontsize=12)
        ax4.grid(axis='x', alpha=0.3)
        ax4.invert_yaxis()
    
    # Target 2: MAE
    ax5 = axes[1, 1]
    if len(df_t2) > 0:
        df_t2_sorted_mae = df_t2.sort_values('MAE', ascending=True).head(15)
        ax5.barh(range(len(df_t2_sorted_mae)), df_t2_sorted_mae['MAE'], color=colors[:len(df_t2_sorted_mae)])
        ax5.set_yticks(range(len(df_t2_sorted_mae)))
        ax5.set_yticklabels([f"{row['Model']} ({row['Dataset']})" for _, row in df_t2_sorted_mae.iterrows()], fontsize=8)
        ax5.set_xlabel('MAE', fontweight='bold')
        ax5.set_title('Specific Grinding Energy - MAE Comparison', fontweight='bold', fontsize=12)
        ax5.grid(axis='x', alpha=0.3)
        ax5.invert_yaxis()
    
    # Target 2: Composite Score
    ax6 = axes[1, 2]
    if len(df_t2) > 0:
        df_t2_sorted_comp = df_t2.sort_values('Composite_Score', ascending=False).head(15)
        ax6.barh(range(len(df_t2_sorted_comp)), df_t2_sorted_comp['Composite_Score'], color=colors[:len(df_t2_sorted_comp)])
        ax6.set_yticks(range(len(df_t2_sorted_comp)))
        ax6.set_yticklabels([f"{row['Model']} ({row['Dataset']})" for _, row in df_t2_sorted_comp.iterrows()], fontsize=8)
        ax6.set_xlabel('Composite Score', fontweight='bold')
        ax6.set_title('Specific Grinding Energy - Composite Score', fontweight='bold', fontsize=12)
        ax6.grid(axis='x', alpha=0.3)
        ax6.invert_yaxis()
    
    plt.tight_layout()
    plt.savefig('comprehensive_model_performance_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Performance comparison visualization saved!")
else:
    print("⚠️  No metrics available for visualization.")


## Step 4: Feature Importance Aggregation

This section aggregates feature importance from multiple sources:
- **SHAP values** from all models
- **Feature importances** from tree-based models (RF, XGBoost)
- **Correlation values** (Spearman and Pearson)
- Creates a unified ranking of feature importance


In [ ]:
# Collect all feature importance data from SHAP, feature_importances_, and correlations
all_features = [
    "Moisture content (% wb)",
    "Initial avg. particle size, (mm)",
    "Feed rate (g/s)",
    "Motor size, kW",
    "Screen size, mm",
    "Grinder type",
    "Biomass"
]

def collect_feature_importance():
    """Collect feature importance from all available sources."""
    
    results = {feat: {} for feat in all_features}
    
    # Try to collect SHAP values
    shap_sources = [
        ('SHAP_RF_Original_T1', 'shap_values_rf_t1', 'X'),
        ('SHAP_RF_Original_T2', 'shap_values_rf_t2', 'X'),
        ('SHAP_ANN_Original_T1', 'shap_values_ann_t1', 'X'),
        ('SHAP_ANN_Original_T2', 'shap_values_ann_t2', 'X'),
        ('SHAP_XGB_Original_T1', 'shap_values_xgb_t1_values', 'X'),
        ('SHAP_XGB_Original_T2', 'shap_values_xgb_t2_values', 'X'),
        ('SHAP_ANN_Spearman_T1', 'shap_values_ann_spearman_t1', 'X_train_spearman'),
        ('SHAP_ANN_Spearman_T2', 'shap_values_ann_spearman_t2', 'X_train_spearman'),
        ('SHAP_ANN_Pearson_T1', 'shap_values_ann_pearson_t1', 'X_train_pearson'),
        ('SHAP_ANN_Pearson_T2', 'shap_values_ann_pearson_t2', 'X_train_pearson'),
        ('SHAP_ANN_Imputed_T1', 'shap_values_ann_imputed_t1', 'X_train_imputed'),
        ('SHAP_ANN_Imputed_T2', 'shap_values_ann_imputed_t2', 'X_train_imputed'),
    ]
    
    for name, var_name, feature_var in shap_sources:
        try:
            if var_name in globals() and feature_var in globals():
                shap_vals = globals()[var_name]
                feature_names = globals()[feature_var].columns.tolist() if hasattr(globals()[feature_var], 'columns') else []
                
                if isinstance(shap_vals, np.ndarray) and len(feature_names) > 0:
                    shap_mean = np.abs(shap_vals).mean(axis=0) if len(shap_vals.shape) > 1 else np.abs(shap_vals)
                    for i, feat in enumerate(feature_names):
                        if feat in all_features and i < len(shap_mean):
                            results[feat][name] = float(shap_mean[i])
        except Exception as e:
            pass
    
    # Try to collect feature importances from tree models
    fi_sources = [
        ('FI_RF_Original_T1', 'rf_best_t1', 'X'),
        ('FI_RF_Original_T2', 'rf_best_t2', 'X'),
        ('FI_XGB_Original_T1', 'xgb_best_t1', 'X'),
        ('FI_XGB_Original_T2', 'xgb_best_t2', 'X'),
    ]
    
    for name, model_var, feature_var in fi_sources:
        try:
            if model_var in globals() and feature_var in globals():
                model = globals()[model_var]
                feature_names = globals()[feature_var].columns.tolist() if hasattr(globals()[feature_var], 'columns') else []
                
                if hasattr(model, 'feature_importances_') and len(feature_names) > 0:
                    importances = model.feature_importances_
                    for i, feat in enumerate(feature_names):
                        if feat in all_features and i < len(importances):
                            results[feat][name] = float(importances[i])
        except Exception as e:
            pass
    
    # Try to collect correlations
    corr_sources = [
        ('Corr_Spearman_T1', 'target1_corr_spearman', None),
        ('Corr_Pearson_T1', 'target1_corr_pearson', None),
        ('Corr_Spearman_T2', 'target2_corr_spearman', None),
        ('Corr_Pearson_T2', 'target2_corr_pearson', None),
    ]
    
    for name, corr_var, _ in corr_sources:
        try:
            if corr_var in globals():
                corr_series = globals()[corr_var]
                if isinstance(corr_series, pd.Series):
                    for feat in corr_series.index:
                        if feat in all_features:
                            results[feat][name] = float(abs(corr_series[feat]))
        except Exception as e:
            pass
    
    # Convert to DataFrame
    df_importance = pd.DataFrame(results).T
    df_importance = df_importance.fillna(0)
    
    return df_importance

# Collect feature importance
df_feature_importance = collect_feature_importance()

print("=" * 80)
print("FEATURE IMPORTANCE DATA COLLECTED")
print("=" * 80)
print(f"\nFeatures analyzed: {len(df_feature_importance)}")
print(f"Importance methods: {len(df_feature_importance.columns)}")
if len(df_feature_importance.columns) > 0:
    print("\nFeature Importance DataFrame:")
    print(df_feature_importance)
else:
    print("\n⚠️  No feature importance data found. Please ensure the original notebook has been executed.")


In [ ]:
# Calculate aggregated feature importance scores
if len(df_feature_importance.columns) > 0:
    def calculate_aggregated_importance(df_imp):
        """Calculate aggregated importance scores across all methods."""
        
        df_agg = df_imp.copy()
        
        # Group by method type
        shap_cols = [col for col in df_agg.columns if 'SHAP' in col]
        fi_cols = [col for col in df_agg.columns if 'FI_' in col]
        corr_cols = [col for col in df_agg.columns if 'Corr' in col]
        
        # Target 1 and Target 2 columns
        t1_cols = [col for col in df_agg.columns if '_T1' in col]
        t2_cols = [col for col in df_agg.columns if '_T2' in col]
        
        # Aggregate by method type
        if shap_cols:
            df_agg['SHAP_Mean'] = df_agg[shap_cols].mean(axis=1)
            df_agg['SHAP_Std'] = df_agg[shap_cols].std(axis=1)
        
        if fi_cols:
            df_agg['FI_Mean'] = df_agg[fi_cols].mean(axis=1)
            df_agg['FI_Std'] = df_agg[fi_cols].std(axis=1)
        
        if corr_cols:
            df_agg['Corr_Mean'] = df_agg[corr_cols].mean(axis=1)
            df_agg['Corr_Std'] = df_agg[corr_cols].std(axis=1)
        
        # Aggregate by target
        if t1_cols:
            df_agg['Target1_Mean'] = df_agg[t1_cols].mean(axis=1)
            df_agg['Target1_Std'] = df_agg[t1_cols].std(axis=1)
        
        if t2_cols:
            df_agg['Target2_Mean'] = df_agg[t2_cols].mean(axis=1)
            df_agg['Target2_Std'] = df_agg[t2_cols].std(axis=1)
        
        # Overall aggregate (normalize each method first)
        method_scores = []
        
        if 'SHAP_Mean' in df_agg.columns:
            shap_norm = (df_agg['SHAP_Mean'] - df_agg['SHAP_Mean'].min()) / (df_agg['SHAP_Mean'].max() - df_agg['SHAP_Mean'].min() + 1e-10)
            method_scores.append(shap_norm)
        
        if 'FI_Mean' in df_agg.columns:
            fi_norm = (df_agg['FI_Mean'] - df_agg['FI_Mean'].min()) / (df_agg['FI_Mean'].max() - df_agg['FI_Mean'].min() + 1e-10)
            method_scores.append(fi_norm)
        
        if 'Corr_Mean' in df_agg.columns:
            corr_norm = (df_agg['Corr_Mean'] - df_agg['Corr_Mean'].min()) / (df_agg['Corr_Mean'].max() - df_agg['Corr_Mean'].min() + 1e-10)
            method_scores.append(corr_norm)
        
        if method_scores:
            df_agg['Overall_Importance'] = pd.concat(method_scores, axis=1).mean(axis=1)
        
        # Rank features
        if 'Overall_Importance' in df_agg.columns:
            df_agg['Overall_Rank'] = df_agg['Overall_Importance'].rank(ascending=False)
        
        if 'Target1_Mean' in df_agg.columns:
            df_agg['Target1_Rank'] = df_agg['Target1_Mean'].rank(ascending=False)
        
        if 'Target2_Mean' in df_agg.columns:
            df_agg['Target2_Rank'] = df_agg['Target2_Mean'].rank(ascending=False)
        
        return df_agg
    
    df_importance_aggregated = calculate_aggregated_importance(df_feature_importance)
    
    print("=" * 80)
    print("AGGREGATED FEATURE IMPORTANCE")
    print("=" * 80)
    print("\nOverall Feature Importance Ranking:")
    if 'Overall_Importance' in df_importance_aggregated.columns:
        df_ranked = df_importance_aggregated.sort_values('Overall_Importance', ascending=False)
        display_cols = ['Overall_Importance', 'Overall_Rank']
        if 'Target1_Mean' in df_ranked.columns:
            display_cols.append('Target1_Mean')
        if 'Target2_Mean' in df_ranked.columns:
            display_cols.append('Target2_Mean')
        print(df_ranked[display_cols].to_string())
    else:
        print(df_importance_aggregated.to_string())
else:
    print("⚠️  No feature importance data available for aggregation.")


## Step 5: Comprehensive Feature Importance Visualization

Visualizations showing:
- Overall feature importance ranking
- Feature importance by target variable
- Comparison across different methods (SHAP, Feature Importance, Correlation)
- Heatmap of all importance values


In [ ]:
# Create comprehensive feature importance visualizations
if len(df_feature_importance.columns) > 0 and 'Overall_Importance' in df_importance_aggregated.columns:
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    
    # 1. Overall Feature Importance (Bar Plot)
    ax1 = axes[0, 0]
    df_sorted = df_importance_aggregated.sort_values('Overall_Importance', ascending=True)
    ax1.barh(range(len(df_sorted)), df_sorted['Overall_Importance'], color=colors[:len(df_sorted)])
    ax1.set_yticks(range(len(df_sorted)))
    ax1.set_yticklabels(df_sorted.index, fontsize=10)
    ax1.set_xlabel('Overall Importance Score (Normalized)', fontsize=11, fontweight='bold')
    ax1.set_title('Overall Feature Importance Ranking\n(Combined: SHAP + Feature Importance + Correlation)', 
                 fontsize=12, fontweight='bold')
    ax1.grid(axis='x', alpha=0.3)
    
    # 2. Target 1 vs Target 2 Importance
    ax2 = axes[0, 1]
    if 'Target1_Mean' in df_importance_aggregated.columns and 'Target2_Mean' in df_importance_aggregated.columns:
        x_pos = np.arange(len(df_importance_aggregated))
        width = 0.35
        ax2.barh(x_pos - width/2, df_importance_aggregated['Target1_Mean'], width, 
                label='Final Particle Size', color=colors[0], alpha=0.8)
        ax2.barh(x_pos + width/2, df_importance_aggregated['Target2_Mean'], width, 
                label='Specific Grinding Energy', color=colors[1], alpha=0.8)
        ax2.set_yticks(x_pos)
        ax2.set_yticklabels(df_importance_aggregated.index, fontsize=10)
        ax2.set_xlabel('Mean Importance Score', fontsize=11, fontweight='bold')
        ax2.set_title('Feature Importance by Target Variable', fontsize=12, fontweight='bold')
        ax2.legend(fontsize=10)
        ax2.grid(axis='x', alpha=0.3)
    
    # 3. Method Comparison (SHAP vs FI vs Correlation)
    ax3 = axes[1, 0]
    method_cols = []
    method_labels = []
    if 'SHAP_Mean' in df_importance_aggregated.columns:
        method_cols.append('SHAP_Mean')
        method_labels.append('SHAP')
    if 'FI_Mean' in df_importance_aggregated.columns:
        method_cols.append('FI_Mean')
        method_labels.append('Feature Importance')
    if 'Corr_Mean' in df_importance_aggregated.columns:
        method_cols.append('Corr_Mean')
        method_labels.append('Correlation')
    
    if method_cols:
        x_pos = np.arange(len(df_importance_aggregated))
        width = 0.25
        for i, (col, label) in enumerate(zip(method_cols, method_labels)):
            values = df_importance_aggregated[col]
            values_norm = (values - values.min()) / (values.max() - values.min() + 1e-10)
            ax3.barh(x_pos + i*width - width*(len(method_cols)-1)/2, values_norm, width, 
                    label=label, color=colors[i], alpha=0.8)
        ax3.set_yticks(x_pos)
        ax3.set_yticklabels(df_importance_aggregated.index, fontsize=10)
        ax3.set_xlabel('Normalized Importance Score', fontsize=11, fontweight='bold')
        ax3.set_title('Feature Importance by Method\n(Normalized for Comparison)', fontsize=12, fontweight='bold')
        ax3.legend(fontsize=10)
        ax3.grid(axis='x', alpha=0.3)
    
    # 4. Heatmap of all importance methods
    ax4 = axes[1, 1]
    heatmap_cols = [col for col in df_feature_importance.columns if df_feature_importance[col].dtype in [np.float64, np.int64]]
    if heatmap_cols:
        df_heatmap = df_feature_importance[heatmap_cols].copy()
        for col in df_heatmap.columns:
            col_max = df_heatmap[col].max()
            if col_max > 0:
                df_heatmap[col] = df_heatmap[col] / col_max
        
        sns.heatmap(df_heatmap.T, annot=True, fmt='.2f', cmap='YlOrRd', 
                    cbar_kws={'label': 'Normalized Importance'}, ax=ax4, 
                    yticklabels=[col[:30] for col in df_heatmap.columns],
                    xticklabels=[feat[:20] for feat in df_heatmap.index])
        ax4.set_title('Feature Importance Heatmap\n(All Methods, Normalized)', fontsize=12, fontweight='bold')
        ax4.set_xlabel('Features', fontsize=11, fontweight='bold')
        ax4.set_ylabel('Importance Methods', fontsize=11, fontweight='bold')
        plt.setp(ax4.get_xticklabels(), rotation=45, ha='right')
        plt.setp(ax4.get_yticklabels(), rotation=0)
    
    plt.tight_layout()
    plt.savefig('comprehensive_feature_importance_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Feature importance visualization saved!")
else:
    print("⚠️  Insufficient data for feature importance visualization.")


## Step 6: Statistical Analysis and Final Recommendations

This section provides:
- Statistical summaries by model type, dataset, and feature set
- Feature importance rankings and consistency analysis
- Final recommendations for best models and most important features


In [ ]:
# Generate final summary and recommendations
print("=" * 80)
print("FINAL ANALYSIS SUMMARY AND RECOMMENDATIONS")
print("=" * 80)

if len(df_metrics_normalized) > 0:
    # Best models
    print("\n1. BEST PERFORMING MODELS:")
    print("-" * 80)
    
    if len(rankings_t1) > 0:
        best_t1 = rankings_t1.iloc[0]
        print(f"\nFinal Particle Size:")
        print(f"  Model: {best_t1['Model']}")
        print(f"  Dataset: {best_t1['Dataset']}")
        print(f"  Features: {best_t1['Features']}")
        print(f"  R²: {best_t1['R2']:.4f}")
        print(f"  MAE: {best_t1['MAE']:.4f}")
        print(f"  RMSE: {best_t1['RMSE']:.4f}")
    
    if len(rankings_t2) > 0:
        best_t2 = rankings_t2.iloc[0]
        print(f"\nSpecific Grinding Energy:")
        print(f"  Model: {best_t2['Model']}")
        print(f"  Dataset: {best_t2['Dataset']}")
        print(f"  Features: {best_t2['Features']}")
        print(f"  R²: {best_t2['R2']:.4f}")
        print(f"  MAE: {best_t2['MAE']:.4f}")
        print(f"  RMSE: {best_t2['RMSE']:.4f}")
    
    # Key insights
    print("\n\n2. KEY INSIGHTS:")
    print("-" * 80)
    
    # Model type performance
    model_perf = df_metrics_normalized.groupby(df_metrics_normalized['Model'].str.split(' ').str[0])['R2'].mean().sort_values(ascending=False)
    print(f"\nAverage R² by Model Type:")
    for model_type, r2 in model_perf.items():
        print(f"  {model_type}: {r2:.4f}")
    
    # Dataset performance
    dataset_perf = df_metrics_normalized.groupby('Dataset')['R2'].mean().sort_values(ascending=False)
    print(f"\nAverage R² by Dataset:")
    for dataset, r2 in dataset_perf.items():
        print(f"  {dataset}: {r2:.4f}")
    
    # Feature set performance
    feature_perf = df_metrics_normalized.groupby('Features')['R2'].mean().sort_values(ascending=False)
    print(f"\nAverage R² by Feature Set:")
    for features, r2 in feature_perf.items():
        print(f"  {features}: {r2:.4f}")

# Most important features
if len(df_feature_importance.columns) > 0 and 'Overall_Importance' in df_importance_aggregated.columns:
    print("\n\n3. MOST IMPORTANT FEATURES:")
    print("-" * 80)
    top_features = df_importance_aggregated.nlargest(5, 'Overall_Importance')
    print("\nTop 5 Features (Overall):")
    for i, (feature, row) in enumerate(top_features.iterrows(), 1):
        print(f"  {i}. {feature}: {row['Overall_Importance']:.4f}")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)


## Step 7: Export Results

Export all analysis results to CSV files for further use.


In [ ]:
# Export all results to CSV files
if len(df_metrics_normalized) > 0:
    df_metrics_normalized.to_csv('comprehensive_model_performance.csv', index=False)
    print("✓ Model performance metrics exported to 'comprehensive_model_performance.csv'")
    
    if len(rankings_t1) > 0:
        rankings_t1.to_csv('model_rankings_final_particle_size.csv', index=False)
        print("✓ Model rankings for Final Particle Size exported")
    
    if len(rankings_t2) > 0:
        rankings_t2.to_csv('model_rankings_specific_grinding_energy.csv', index=False)
        print("✓ Model rankings for Specific Grinding Energy exported")

if len(df_feature_importance.columns) > 0:
    if 'Overall_Importance' in df_importance_aggregated.columns:
        df_importance_aggregated.to_csv('comprehensive_feature_importance.csv')
        print("✓ Feature importance data exported to 'comprehensive_feature_importance.csv'")
    else:
        df_feature_importance.to_csv('comprehensive_feature_importance.csv')
        print("✓ Feature importance data exported to 'comprehensive_feature_importance.csv'")

print("\n" + "=" * 80)
print("ALL RESULTS EXPORTED SUCCESSFULLY")
print("=" * 80)
